## ScanObjectNN Zero-Shot Evaluation

In [2]:
# ============================================
# Cell 1 — Imports and paths
# ============================================
import os, json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

REPO_ROOT   = Path('/Users/dosvatsky/3D Object Detection')
DATA_DIR    = REPO_ROOT / 'data' / 'scanobjectnn'
CKPT_DIR    = REPO_ROOT / 'checkpoints'
RESULTS_DIR = REPO_ROOT

DATA_DIR.mkdir(parents=True, exist_ok=True)

device = (torch.device('cuda') if torch.cuda.is_available() else
          torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cpu'))
print(f'Using device: {device}')
print(f'Repo root:    {REPO_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Checkpoints:  {CKPT_DIR}')

Using device: mps
Repo root:    /Users/dosvatsky/3D Object Detection
Data dir:     /Users/dosvatsky/3D Object Detection/data/scanobjectnn
Checkpoints:  /Users/dosvatsky/3D Object Detection/checkpoints


In [3]:
# ============================================
# Cell 2 — Verify ScanObjectNN files exist
# ============================================
files_to_check = {
    'OBJ_BG':    DATA_DIR / 'test_objectdataset.h5',
    'PB_T50_RS': DATA_DIR / 'test_objectdataset_augmentedrot_scale75.h5',
}

all_ok = True
for name, path in files_to_check.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f'OK       {name:10s}  {path.name}  ({size_mb:.1f} MB)')
    else:
        print(f'MISSING  {name:10s}  -> download to  {path}')
        all_ok = False

if not all_ok:
    print('\nDownload the missing files, then re-run this cell before continuing.')
else:
    print('\nBoth files present — ready to load.')

OK       OBJ_BG      test_objectdataset.h5  (11.4 MB)
OK       PB_T50_RS   test_objectdataset_augmentedrot_scale75.h5  (60.1 MB)

Both files present — ready to load.


## loading HDF5 and inspecting class distribution

In [4]:
# ============================================
# Cell 5 — Load HDF5 and inspect class distribution
# ============================================
SCANOBJ_CLASSES = [
    'bag', 'bin', 'box', 'cabinet', 'chair', 'desk', 'display', 'door',
    'shelf', 'table', 'bed', 'pillow', 'sink', 'sofa', 'toilet'
]

def load_scanobjectnn(h5_path):
    with h5py.File(h5_path, 'r') as f:
        data   = f['data'][:].astype(np.float32)   # (N, 2048, 3)
        labels = f['label'][:].astype(np.int64)    # (N,)
    return data, labels

for name, path in files_to_check.items():
    if not path.exists():
        print(f'[skip] {name}: file not present')
        continue
    data, labels = load_scanobjectnn(path)
    print(f'\n=== {name} ===')
    print(f'  data shape:  {data.shape}')
    print(f'  label range: {labels.min()} to {labels.max()}')
    print(f'  n samples:   {len(labels)}')
    print(f'  class distribution:')
    unique, counts = np.unique(labels, return_counts=True)
    for u, c in zip(unique, counts):
        print(f'    {u:2d} {SCANOBJ_CLASSES[u]:10s}: {c:4d}')


=== OBJ_BG ===
  data shape:  (581, 2048, 3)
  label range: 0 to 14
  n samples:   581
  class distribution:
     0 bag       :   17
     1 bin       :   40
     2 box       :   28
     3 cabinet   :   75
     4 chair     :   78
     5 desk      :   30
     6 display   :   42
     7 door      :   42
     8 shelf     :   49
     9 table     :   54
    10 bed       :   22
    11 pillow    :   21
    12 sink      :   24
    13 sofa      :   42
    14 toilet    :   17

=== PB_T50_RS ===
  data shape:  (2882, 2048, 3)
  label range: 0 to 14
  n samples:   2882
  class distribution:
     0 bag       :   83
     1 bin       :  199
     2 box       :  133
     3 cabinet   :  372
     4 chair     :  390
     5 desk      :  150
     6 display   :  204
     7 door      :  210
     8 shelf     :  241
     9 table     :  270
    10 bed       :  110
    11 pillow    :  105
    12 sink      :  120
    13 sofa      :  210
    14 toilet    :   85


## MSG PointNet++ Architecture Definition

In [5]:
# ============================================
# Cell 7 — PointNet++ MSG model definition
# ============================================
# ===== FPS, Ball Query, index_points =====
def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    dev = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    distance = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch_idx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(npoint):
        centroids[:, i] = farthest
        cxyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - cxyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    B = points.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0] = 1
    bi = torch.arange(B, dtype=torch.long, device=points.device).view(vs).repeat(rs)
    return points[bi, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    dev = xyz.device
    gi = torch.arange(N, dtype=torch.long, device=dev).view(1, 1, N).repeat(B, S, 1)
    sd = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    gi[sd > radius ** 2] = N
    gi = gi.sort(dim=-1)[0][:, :, :nsample]
    gf = gi[:, :, 0:1].repeat(1, 1, nsample); gi[gi == N] = gf[gi == N]
    return gi


class SetAbstractionMSG(nn.Module):
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        self.npoint, self.radii, self.nsamples = npoint, radii, nsamples
        self.conv_blocks, self.bn_blocks = nn.ModuleList(), nn.ModuleList()
        for mlp in mlps:
            convs, bns = nn.ModuleList(), nn.ModuleList()
            last = in_channel + 3
            for c in mlp:
                convs.append(nn.Conv2d(last, c, 1)); bns.append(nn.BatchNorm2d(c)); last = c
            self.conv_blocks.append(convs); self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        fps = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps); outs = []
        for i, (r, k) in enumerate(zip(self.radii, self.nsamples)):
            nn_idx = ball_query(r, k, xyz, new_xyz)
            g_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
            if features is not None:
                g = torch.cat([g_xyz, index_points(features, nn_idx)], dim=-1)
            else:
                g = g_xyz
            g = g.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                g = F.relu(bn(conv(g)))
            outs.append(g.max(dim=-1)[0])
        return new_xyz, torch.cat(outs, dim=1).permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for c in mlp:
            self.convs.append(nn.Conv1d(last, c, 1)); self.bns.append(nn.BatchNorm1d(c)); last = c

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


class PointNetPlusPlusMSG(nn.Module):
    def __init__(self, num_classes=40, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(512, [0.1, 0.2, 0.4], [16, 32, 128], 0,
                                     [[32, 32, 64], [64, 64, 128], [64, 96, 128]])
        self.sa2 = SetAbstractionMSG(128, [0.2, 0.4, 0.8], [32, 64, 128], 320,
                                     [[64, 64, 128], [128, 128, 256], [128, 128, 256]])
        self.sa_global = GlobalSetAbstraction(640 + 3, [256, 512, 1024])
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_f = self.sa1(xyz, None)
        l2_xyz, l2_f = self.sa2(l1_xyz, l1_f)
        return self.classifier(self.sa_global(l2_xyz, l2_f))


print("PointNet++ MSG architecture defined.")

PointNet++ MSG architecture defined.


## Checkpoint Loading And Smoke Test

In [8]:
# ============================================
# Cell 9 — Checkpoint loading helper + smoke test (FIXED)
# ============================================
def load_msg_model(ckpt_path, device, num_classes=40):
    '''Load a PointNet++ MSG checkpoint into a fresh model.
       Tries common wrapper keys, validates that weights actually loaded.'''
    model = PointNetPlusPlusMSG(num_classes=num_classes).to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # Diagnostic — show what is actually inside the checkpoint
    if isinstance(ckpt, dict):
        print(f'    Top-level keys in checkpoint: {list(ckpt.keys())}')

    # Priority list — EMA variants first (usually better eval), then regular model
    candidate_keys = (
        'ema_state_dict', 'ema_model', 'model_ema', 'ema',
        'model_state_dict', 'state_dict', 'model',
    )

    state = None
    if isinstance(ckpt, dict):
        for key in candidate_keys:
            if key in ckpt and isinstance(ckpt[key], dict):
                state = ckpt[key]
                print(f'    Using key [{key}] for weights')
                break

    if state is None:
        state = ckpt
        print(f'    Using raw checkpoint dict as state-dict')

    # Strip 'module.' prefix if checkpoint was saved from DataParallel
    state = {k.replace('module.', '', 1) if k.startswith('module.') else k: v
             for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)

    n_total  = len(list(model.state_dict().keys()))
    n_loaded = n_total - len(missing)
    print(f'    Loaded {n_loaded}/{n_total} weight tensors')

    if n_loaded < n_total * 0.9:
        raise RuntimeError(
            f'ONLY {n_loaded}/{n_total} weights loaded — model would be ~random!\n'
            f'  Unexpected keys (first 5): {unexpected[:5]}\n'
            f'  Missing keys     (first 5): {missing[:5]}\n'
            f'  Fix: inspect checkpoint structure and add the right key to candidate_keys.'
        )

    model.eval()
    return model


# Smoke test — load each checkpoint, confirm weights actually loaded
print('Smoke test — loading each checkpoint:\n')
for name, path in CKPT_FILES.items():
    if not path.exists():
        continue
    print(f'[{name}]')
    m = load_msg_model(path, device)
    n_params = sum(p.numel() for p in m.parameters())
    print(f'    OK — {n_params/1e6:.2f}M params\n')
    del m
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('All checkpoints loaded successfully.')

Smoke test — loading each checkpoint:

[original]
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors
    OK — 1.75M params

[domain_aug]
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors
    OK — 1.75M params

[finetuned]
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc']
    Using key [model] for weights
    Loaded 163/163 weight tensors
    OK — 1.75M params

All checkpoints loaded successfully.


## FPS and  normalization to ModelNet40

In [9]:
# ============================================
# Cell 10 — Normalization (match ModelNet40 pre-processing)
# ============================================
def fps_torch(points, n_samples):
    '''Pure-PyTorch farthest point sampling on a single cloud.
       points: (N, 3) -> returns (n_samples, 3).'''
    N = points.shape[0]
    if N <= n_samples:
        # Too few points — pad by random repetition
        idx = torch.cat([torch.arange(N, device=points.device),
                         torch.randint(0, N, (n_samples - N,), device=points.device)])
        return points[idx]
    centroids = torch.zeros(n_samples, dtype=torch.long, device=points.device)
    dist      = torch.full((N,), 1e10, device=points.device)
    farthest  = torch.randint(0, N, (1,), device=points.device).item()
    for i in range(n_samples):
        centroids[i] = farthest
        c = points[farthest].unsqueeze(0)
        d = ((points - c) ** 2).sum(dim=-1)
        dist     = torch.minimum(dist, d)
        farthest = torch.argmax(dist).item()
    return points[centroids]


def normalize_to_modelnet40(points_np, n_target=1024, device='cpu'):
    '''Same pre-processing as ModelNet40 training:
       1. FPS-downsample (or pad) to n_target points
       2. Center at origin
       3. Scale to unit sphere
    '''
    points = torch.from_numpy(points_np).float().to(device)
    if points.shape[0] != n_target:
        points = fps_torch(points, n_target)
    points = points - points.mean(dim=0, keepdim=True)
    max_dist = points.norm(dim=1).max()
    points   = points / (max_dist + 1e-8)
    return points


# Sanity check
_test = np.random.randn(2048, 3).astype(np.float32)
_norm = normalize_to_modelnet40(_test, n_target=1024, device='cpu')
print(f'Test crop shape:  {_norm.shape}')
print(f'Max radius:       {_norm.norm(dim=1).max().item():.4f}  (should be ~1.0)')
print(f'Centroid offset:  {_norm.mean(dim=0).abs().sum().item():.6f}  (should be ~0.0)')

Test crop shape:  torch.Size([1024, 3])
Max radius:       1.0000  (should be ~1.0)
Centroid offset:  0.000000  (should be ~0.0)


## class mapping ScanObjectNN to ModelNet40

In [10]:
# ============================================
# Cell 11 — Class mapping ScanObjectNN -> ModelNet40
# ============================================
MODELNET40_CLASSES = [
    'airplane', 'bathtub', 'bed', 'bench', 'bookshelf', 'bottle', 'bowl', 'car',
    'chair', 'cone', 'cup', 'curtain', 'desk', 'door', 'dresser', 'flower_pot',
    'glass_box', 'guitar', 'keyboard', 'lamp', 'laptop', 'mantel', 'monitor',
    'night_stand', 'person', 'piano', 'plant', 'radio', 'range_hood', 'sink',
    'sofa', 'stairs', 'stool', 'table', 'tent', 'toilet', 'tv_stand', 'vase',
    'wardrobe', 'xbox'
]
MN40_NAME_TO_IDX = {n: i for i, n in enumerate(MODELNET40_CLASSES)}

# STRICT — only the obvious twin counts
STRICT_MAP = {
    'chair':   {'chair'},
    'desk':    {'desk'},
    'door':    {'door'},
    'table':   {'table'},
    'bed':     {'bed'},
    'sink':    {'sink'},
    'sofa':    {'sofa'},
    'toilet':  {'toilet'},
    'shelf':   {'bookshelf'},
    'cabinet': {'wardrobe'},
    'display': {'monitor'},
}
# Unmappable ScanObjectNN classes — no ModelNet40 equivalent:
#   bag, bin, box, pillow  (model never trained on these, will always be wrong)

# PERMISSIVE — accept visually-similar MN40 classes
PERMISSIVE_MAP = {
    'chair':   {'chair', 'stool', 'bench'},
    'desk':    {'desk', 'table'},
    'door':    {'door', 'curtain'},
    'table':   {'table', 'desk'},
    'bed':     {'bed'},
    'sink':    {'sink'},
    'sofa':    {'sofa', 'bench'},
    'toilet':  {'toilet'},
    'shelf':   {'bookshelf', 'tv_stand', 'wardrobe'},
    'cabinet': {'wardrobe', 'night_stand', 'dresser'},
    'display': {'monitor'},
}

def names_to_idx(name_set):
    return {MN40_NAME_TO_IDX[n] for n in name_set if n in MN40_NAME_TO_IDX}

STRICT_MAP_IDX     = {k: names_to_idx(v) for k, v in STRICT_MAP.items()}
PERMISSIVE_MAP_IDX = {k: names_to_idx(v) for k, v in PERMISSIVE_MAP.items()}

print(f'Mappable ScanObjectNN classes ({len(STRICT_MAP)} / 15):\n')
print(f'  {"ScanObjectNN":12s}  {"Strict":35s}  {"Permissive":40s}')
print(f'  {"-"*12}  {"-"*35}  {"-"*40}')
for k in STRICT_MAP:
    print(f'  {k:12s}  {str(sorted(STRICT_MAP[k])):35s}  {str(sorted(PERMISSIVE_MAP[k])):40s}')
print(f'\nUnmappable (will always be marked wrong): bag, bin, box, pillow')

Mappable ScanObjectNN classes (11 / 15):

  ScanObjectNN  Strict                               Permissive                              
  ------------  -----------------------------------  ----------------------------------------
  chair         ['chair']                            ['bench', 'chair', 'stool']             
  desk          ['desk']                             ['desk', 'table']                       
  door          ['door']                             ['curtain', 'door']                     
  table         ['table']                            ['desk', 'table']                       
  bed           ['bed']                              ['bed']                                 
  sink          ['sink']                             ['sink']                                
  sofa          ['sofa']                             ['bench', 'sofa']                       
  toilet        ['toilet']                           ['toilet']                              
  shelf         ['

## Inference Loop


In [11]:
# ============================================
# Cell 12 — Inference loop
# ============================================
@torch.no_grad()
def run_inference(model, data, device, batch_size=32, n_points=1024):
    '''Run inference on N point clouds.
       data: (N, 2048, 3) numpy array.
       Returns: predictions (N,) and confidences (N,).'''
    model.eval()
    N = len(data)
    all_preds, all_confs = [], []

    for i in tqdm(range(0, N, batch_size), desc='Inference'):
        batch_np = data[i:i+batch_size]
        batch_tensors = [
            normalize_to_modelnet40(pc, n_target=n_points, device='cpu')
            for pc in batch_np
        ]
        batch = torch.stack(batch_tensors).to(device)        # (B, n_points, 3)

        logits = model(batch)                                 # (B, 40)
        probs  = F.softmax(logits, dim=-1)
        preds  = probs.argmax(dim=-1).cpu().numpy()
        confs  = probs.max(dim=-1).values.cpu().numpy()
        all_preds.append(preds)
        all_confs.append(confs)

    return np.concatenate(all_preds), np.concatenate(all_confs)


print('Inference function ready.')

Inference function ready.


## Computational Helpers


In [12]:
# ============================================
# Cell 13 — Accuracy helpers (overall + per-class)
# ============================================
def compute_accuracy(true_scanobj, predicted_mn40, mapping_idx):
    '''
    true_scanobj:    (N,) array of ScanObjectNN class indices (0-14)
    predicted_mn40:  (N,) array of model-predicted MN40 class indices (0-39)
    mapping_idx:     dict {scanobj_class_name -> set of accepted MN40 indices}

    Returns: overall accuracy, per-class dict {name: [n_correct, n_total]}
    '''
    n_correct = 0
    per_class = defaultdict(lambda: [0, 0])

    for t_idx, p_idx in zip(true_scanobj, predicted_mn40):
        name = SCANOBJ_CLASSES[t_idx]
        per_class[name][1] += 1                  # total count for this class
        accepted = mapping_idx.get(name, set())  # which MN40 preds count as correct
        if p_idx in accepted:
            n_correct += 1
            per_class[name][0] += 1              # correct count

    overall = n_correct / len(true_scanobj) if len(true_scanobj) else 0.0
    return overall, dict(per_class)


def print_per_class(per_class, title=''):
    if title:
        print(f'  {title}')
    print(f'    {"class":12s}  {"correct":>8s} / {"total":>5s}   {"acc":>6s}')
    print(f'    {"-"*42}')
    for name in SCANOBJ_CLASSES:
        if name not in per_class:
            continue
        c, t = per_class[name]
        acc = c / t if t else 0
        print(f'    {name:12s}  {c:>8d} / {t:>5d}   {acc:>6.1%}')


print('Accuracy helpers ready.')

Accuracy helpers ready.


## Evaluation

In [13]:
# ============================================
# Cell 14 — Run full evaluation: 3 models × 2 splits
# ============================================
all_results = []
detailed   = {}   # model -> split -> dict (preds, confs, labels, per-class)

for ckpt_name, ckpt_path in CKPT_FILES.items():
    if not ckpt_path.exists():
        print(f'\n[SKIP] {ckpt_name}: checkpoint not found')
        continue
    print(f'\n{"="*60}\nMODEL: {ckpt_name}\n{"="*60}')
    model = load_msg_model(ckpt_path, device)

    detailed[ckpt_name] = {}
    for split_name, h5_path in files_to_check.items():
        if not h5_path.exists():
            print(f'\n  [SKIP] {split_name}: data file not found')
            continue
        print(f'\n  ----- Split: {split_name} -----')
        data, labels = load_scanobjectnn(h5_path)
        preds, confs = run_inference(model, data, device)

        # Overall (counting bag/bin/box/pillow as wrong since they're unmappable)
        strict_all, strict_pc = compute_accuracy(labels, preds, STRICT_MAP_IDX)
        perm_all,   perm_pc   = compute_accuracy(labels, preds, PERMISSIVE_MAP_IDX)

        # Mappable subset only — the fairer sim-to-real number
        mask    = np.array([SCANOBJ_CLASSES[l] in STRICT_MAP for l in labels])
        n_map   = int(mask.sum())
        strict_map, _ = compute_accuracy(labels[mask], preds[mask], STRICT_MAP_IDX)
        perm_map,   _ = compute_accuracy(labels[mask], preds[mask], PERMISSIVE_MAP_IDX)

        print(f'\n  n_total = {len(labels)}   n_mappable = {n_map}')
        print(f'    Strict     (all 15 classes):    {strict_all:.4f}')
        print(f'    Permissive (all 15 classes):    {perm_all:.4f}')
        print(f'    Strict     (mappable only):     {strict_map:.4f}')
        print(f'    Permissive (mappable only):     {perm_map:.4f}')

        print()
        print_per_class(strict_pc, 'Per-class accuracy (strict):')

        all_results.append({
            'model':               ckpt_name,
            'split':               split_name,
            'n_total':             len(labels),
            'n_mappable':          n_map,
            'strict_all':          strict_all,
            'permissive_all':      perm_all,
            'strict_mappable':     strict_map,
            'permissive_mappable': perm_map,
        })
        detailed[ckpt_name][split_name] = {
            'preds':   preds,   'confs': confs,   'labels': labels,
            'strict_pc': strict_pc, 'perm_pc': perm_pc,
        }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\n\nDone with all evaluations.')


MODEL: original
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors

  ----- Split: OBJ_BG -----


Inference: 100%|██████████| 19/19 [11:34<00:00, 36.55s/it]



  n_total = 581   n_mappable = 475
    Strict     (all 15 classes):    0.0706
    Permissive (all 15 classes):    0.1084
    Strict     (mappable only):     0.0863
    Permissive (mappable only):     0.1326

  Per-class accuracy (strict):
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    17     0.0%
    bin                  0 /    40     0.0%
    box                  0 /    28     0.0%
    cabinet              1 /    75     1.3%
    chair               25 /    78    32.1%
    desk                 4 /    30    13.3%
    display              3 /    42     7.1%
    door                 0 /    42     0.0%
    shelf                8 /    49    16.3%
    table                0 /    54     0.0%
    bed                  0 /    22     0.0%
    pillow               0 /    21     0.0%
    sink                 0 /    24     0.0%
    sofa                 0 /    42     0.0%
    toilet               0 /    17     0.0%

  ----- 

Inference: 100%|██████████| 91/91 [20:56<00:00, 13.81s/it] 



  n_total = 2882   n_mappable = 2362
    Strict     (all 15 classes):    0.0531
    Permissive (all 15 classes):    0.0836
    Strict     (mappable only):     0.0648
    Permissive (mappable only):     0.1020

  Per-class accuracy (strict):
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    83     0.0%
    bin                  0 /   199     0.0%
    box                  0 /   133     0.0%
    cabinet              0 /   372     0.0%
    chair              100 /   390    25.6%
    desk                10 /   150     6.7%
    display             16 /   204     7.8%
    door                 1 /   210     0.5%
    shelf               24 /   241    10.0%
    table                0 /   270     0.0%
    bed                  1 /   110     0.9%
    pillow               0 /   105     0.0%
    sink                 0 /   120     0.0%
    sofa                 0 /   210     0.0%
    toilet               1 /    85     1.2%

MODEL:

...


  n_total = 2882   n_mappable = 2362
    Strict     (all 15 classes):    0.0482
    Permissive (all 15 classes):    0.0684
    Strict     (mappable only):     0.0588
    Permissive (mappable only):     0.0834

  Per-class accuracy (strict):
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    83     0.0%
    bin                  0 /   199     0.0%
    box                  0 /   133     0.0%
    cabinet              0 /   372     0.0%
    chair               91 /   390    23.3%
    desk                 5 /   150     3.3%
    display             17 /   204     8.3%
    door                 1 /   210     0.5%
    shelf               16 /   241     6.6%
    table                0 /   270     0.0%
    bed                  4 /   110     3.6%
    pillow               0 /   105     0.0%
    sink                 3 /   120     2.5%
    sofa                 0 /   210     0.0%
    toilet               2 /    85     2.4%

MODEL:

Inference: 100%|██████████| 19/19 [02:09<00:00,  6.83s/it]



  n_total = 581   n_mappable = 475
    Strict     (all 15 classes):    0.0361
    Permissive (all 15 classes):    0.0637
    Strict     (mappable only):     0.0442
    Permissive (mappable only):     0.0779

  Per-class accuracy (strict):
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    17     0.0%
    bin                  0 /    40     0.0%
    box                  0 /    28     0.0%
    cabinet              0 /    75     0.0%
    chair                7 /    78     9.0%
    desk                 1 /    30     3.3%
    display              3 /    42     7.1%
    door                 0 /    42     0.0%
    shelf                7 /    49    14.3%
    table                0 /    54     0.0%
    bed                  2 /    22     9.1%
    pillow               0 /    21     0.0%
    sink                 0 /    24     0.0%
    sofa                 0 /    42     0.0%
    toilet               1 /    17     5.9%

  ----- 

Inference: 100%|██████████| 91/91 [17:43<00:00, 11.68s/it]



  n_total = 2882   n_mappable = 2362
    Strict     (all 15 classes):    0.0257
    Permissive (all 15 classes):    0.0427
    Strict     (mappable only):     0.0313
    Permissive (mappable only):     0.0521

  Per-class accuracy (strict):
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    83     0.0%
    bin                  0 /   199     0.0%
    box                  0 /   133     0.0%
    cabinet              1 /   372     0.3%
    chair               10 /   390     2.6%
    desk                 1 /   150     0.7%
    display             14 /   204     6.9%
    door                 0 /   210     0.0%
    shelf               33 /   241    13.7%
    table                0 /   270     0.0%
    bed                  5 /   110     4.5%
    pillow               0 /   105     0.0%
    sink                 5 /   120     4.2%
    sofa                 3 /   210     1.4%
    toilet               2 /    85     2.4%


Done 

In [14]:
# ============================================
# Cell 16 — Diagnostic: try swapping Y and Z axes
# ============================================
# Hypothesis: ScanObjectNN is Z-up, ModelNet40 is Y-up.
# If true, swapping Y<->Z should massively boost accuracy.

@torch.no_grad()
def run_inference_axis_swapped(model, data, device, batch_size=32, n_points=1024):
    '''Same as run_inference but swaps Y and Z axes of every point cloud.'''
    model.eval()
    all_preds = []
    for i in tqdm(range(0, len(data), batch_size), desc='Inference (Y<->Z)'):
        batch_np = data[i:i+batch_size].copy()
        # Swap Y and Z: [x, y, z] -> [x, z, y]
        batch_np = batch_np[:, :, [0, 2, 1]]
        batch_tensors = [normalize_to_modelnet40(pc, n_target=n_points, device='cpu') for pc in batch_np]
        batch = torch.stack(batch_tensors).to(device)
        logits = model(batch)
        all_preds.append(F.softmax(logits, dim=-1).argmax(dim=-1).cpu().numpy())
    return np.concatenate(all_preds)


# Test on domain_aug + OBJ_BG (smallest split, fastest)
print('Quick axis-swap test on domain_aug + OBJ_BG\n')
model = load_msg_model(CKPT_FILES['domain_aug'], device)
data, labels = load_scanobjectnn(files_to_check['OBJ_BG'])

preds_swapped = run_inference_axis_swapped(model, data, device)
strict_acc, strict_pc   = compute_accuracy(labels, preds_swapped, STRICT_MAP_IDX)
mask = np.array([SCANOBJ_CLASSES[l] in STRICT_MAP for l in labels])
strict_map_acc, _ = compute_accuracy(labels[mask], preds_swapped[mask], STRICT_MAP_IDX)

print(f'\n  Y<->Z swapped, strict all:     {strict_acc:.4f}')
print(f'  Y<->Z swapped, strict mappable: {strict_map_acc:.4f}')
print('\n  Before swap: 0.0826 all / 0.1011 mappable')
print_per_class(strict_pc, '\n  Per-class after swap:')

del model
if device.type == 'cuda':
    torch.cuda.empty_cache()

Quick axis-swap test on domain_aug + OBJ_BG

    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors


Inference (Y<->Z): 100%|██████████| 19/19 [02:28<00:00,  7.83s/it]


  Y<->Z swapped, strict all:     0.3976
  Y<->Z swapped, strict mappable: 0.4863

  Before swap: 0.0826 all / 0.1011 mappable
  
  Per-class after swap:
    class          correct / total      acc
    ------------------------------------------
    bag                  0 /    17     0.0%
    bin                  0 /    40     0.0%
    box                  0 /    28     0.0%
    cabinet              0 /    75     0.0%
    chair               78 /    78   100.0%
    desk                12 /    30    40.0%
    display             26 /    42    61.9%
    door                 4 /    42     9.5%
    shelf               20 /    49    40.8%
    table               36 /    54    66.7%
    bed                 15 /    22    68.2%
    pillow               0 /    21     0.0%
    sink                 8 /    24    33.3%
    sofa                17 /    42    40.5%
    toilet              15 /    17    88.2%


In [15]:
# ============================================
# Cell 17 — Re-run full sweep with Y<->Z axis correction
# ============================================
@torch.no_grad()
def run_inference_corrected(model, data, device, batch_size=32, n_points=1024):
    '''Run inference with Y<->Z axis swap to match ModelNet40 (Y-up) convention.'''
    model.eval()
    all_preds, all_confs = [], []
    for i in tqdm(range(0, len(data), batch_size), desc='Inference (axis-corrected)'):
        batch_np = data[i:i+batch_size].copy()
        batch_np = batch_np[:, :, [0, 2, 1]]      # X, Z, Y  -->  X, Y, Z
        batch_tensors = [normalize_to_modelnet40(pc, n_target=n_points, device='cpu') for pc in batch_np]
        batch = torch.stack(batch_tensors).to(device)
        logits = model(batch)
        probs  = F.softmax(logits, dim=-1)
        all_preds.append(probs.argmax(dim=-1).cpu().numpy())
        all_confs.append(probs.max(dim=-1).values.cpu().numpy())
    return np.concatenate(all_preds), np.concatenate(all_confs)


# Re-run all 6 combinations with axis correction
corrected_results = []
detailed_corrected = {}

for ckpt_name, ckpt_path in CKPT_FILES.items():
    if not ckpt_path.exists():
        continue
    print(f'\n{"="*60}\nMODEL: {ckpt_name} (axis-corrected)\n{"="*60}')
    model = load_msg_model(ckpt_path, device)
    detailed_corrected[ckpt_name] = {}

    for split_name, h5_path in files_to_check.items():
        if not h5_path.exists():
            continue
        print(f'\n  ----- Split: {split_name} -----')
        data, labels = load_scanobjectnn(h5_path)
        preds, confs = run_inference_corrected(model, data, device)

        strict_all, strict_pc = compute_accuracy(labels, preds, STRICT_MAP_IDX)
        perm_all,   perm_pc   = compute_accuracy(labels, preds, PERMISSIVE_MAP_IDX)
        mask = np.array([SCANOBJ_CLASSES[l] in STRICT_MAP for l in labels])
        n_map = int(mask.sum())
        strict_map, _ = compute_accuracy(labels[mask], preds[mask], STRICT_MAP_IDX)
        perm_map,   _ = compute_accuracy(labels[mask], preds[mask], PERMISSIVE_MAP_IDX)

        print(f'\n  Strict     (all 15 classes):    {strict_all:.4f}')
        print(f'  Permissive (all 15 classes):    {perm_all:.4f}')
        print(f'  Strict     (mappable only):     {strict_map:.4f}')
        print(f'  Permissive (mappable only):     {perm_map:.4f}')

        corrected_results.append({
            'model': ckpt_name, 'split': split_name,
            'n_total': len(labels), 'n_mappable': n_map,
            'strict_all': strict_all, 'permissive_all': perm_all,
            'strict_mappable': strict_map, 'permissive_mappable': perm_map,
        })

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

# Save corrected results
corrected_df = pd.DataFrame(corrected_results)
out_csv = RESULTS_DIR / 'scanobjectnn_zero_shot_axis_corrected.csv'
corrected_df.to_csv(out_csv, index=False)

print('\n\nAXIS-CORRECTED RESULTS')
print('=' * 100)
display_df = corrected_df.copy()
for col in ['strict_all', 'permissive_all', 'strict_mappable', 'permissive_mappable']:
    display_df[col] = display_df[col].apply(lambda x: f'{x*100:5.2f}%')
print(display_df.to_string(index=False))
print(f'\nSaved CSV: {out_csv}')


MODEL: original (axis-corrected)
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors

  ----- Split: OBJ_BG -----


Inference (axis-corrected): 100%|██████████| 19/19 [01:42<00:00,  5.41s/it]



  Strict     (all 15 classes):    0.4165
  Permissive (all 15 classes):    0.5112
  Strict     (mappable only):     0.5095
  Permissive (mappable only):     0.6253

  ----- Split: PB_T50_RS -----


Inference (axis-corrected): 100%|██████████| 91/91 [05:42<00:00,  3.76s/it]



  Strict     (all 15 classes):    0.2505
  Permissive (all 15 classes):    0.3147
  Strict     (mappable only):     0.3057
  Permissive (mappable only):     0.3840

MODEL: domain_aug (axis-corrected)
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc', 'is_ema']
    Using key [model] for weights
    Loaded 163/163 weight tensors

  ----- Split: OBJ_BG -----


...


  Strict     (all 15 classes):    0.2588
  Permissive (all 15 classes):    0.3265
  Strict     (mappable only):     0.3158
  Permissive (mappable only):     0.3984

MODEL: finetuned (axis-corrected)
    Top-level keys in checkpoint: ['epoch', 'model', 'val_acc']
    Using key [model] for weights
    Loaded 163/163 weight tensors

  ----- Split: OBJ_BG -----


Inference (axis-corrected): 100%|██████████| 19/19 [01:11<00:00,  3.76s/it]



  Strict     (all 15 classes):    0.4200
  Permissive (all 15 classes):    0.4991
  Strict     (mappable only):     0.5137
  Permissive (mappable only):     0.6105

  ----- Split: PB_T50_RS -----


Inference (axis-corrected): 100%|██████████| 91/91 [06:14<00:00,  4.11s/it]



  Strict     (all 15 classes):    0.2068
  Permissive (all 15 classes):    0.2415
  Strict     (mappable only):     0.2523
  Permissive (mappable only):     0.2947


AXIS-CORRECTED RESULTS
     model     split  n_total  n_mappable strict_all permissive_all strict_mappable permissive_mappable
  original    OBJ_BG      581         475     41.65%         51.12%          50.95%              62.53%
  original PB_T50_RS     2882        2362     25.05%         31.47%          30.57%              38.40%
domain_aug    OBJ_BG      581         475     39.41%         50.60%          48.21%              61.89%
domain_aug PB_T50_RS     2882        2362     25.88%         32.65%          31.58%              39.84%
 finetuned    OBJ_BG      581         475     42.00%         49.91%          51.37%              61.05%
 finetuned PB_T50_RS     2882        2362     20.68%         24.15%          25.23%              29.47%

Saved CSV: /Users/dosvatsky/3D Object Detection/scanobjectnn_zero_shot_axis_corre